# \# 능형회귀

### 다중공선성이 $\hat{\beta}$에 대한 분산과 공분산을 모두 크게한다. 

ref: Montgomery, D. C., Peck, E. A., \& Vining, G. G. (2021). Introduction to linear regression analysis. John Wiley & Sons.

아래와 같이 2개의 regressor가 존재하는 모형을 생각하자. 

$y_i=\beta_0+\beta_1x_{i1}+\beta_2 x_{i2}+\epsilon_i, \quad \epsilon_i ~ \sim iid N(0,\sigma^2)$

편의상 아래를 가정하자. 

$s_y^2=\sum_{i=1}^{n}(y_i-\bar{y})^2=1$

$s_1^2=\sum_{i=1}^{n}(x_{i1}-\bar{x}_1)^2=1$

$s_2^2=\sum_{i=1}^{n}(x_{i2}-\bar{x}_2)^2=1$

최소제곱추정법으로 $\hat{\beta}_1$, $\hat{\beta}_2$를 구하면 아래와 같다. 

$\hat{\beta_1}=\frac{r_{1y}-r_{12}r_{2y}}{1-r_{12}^2}$.

$\hat{\beta_2}=\frac{r_{2y}-r_{12}r_{1y}}{1-r_{12}^2}$.

단, 

$r_{1y}=\frac{s_{1y}}{\sqrt{s_1^2s_y^2}}=s_{1y}$, $\quad s_{1y}=\sum_{i=1}^{n}(y_i-\bar{y})(x_{i1}-\bar{x}_1)$

$r_{2y}=\frac{s_{2y}}{\sqrt{s_2^2s_y^2}}=s_{2y}$, $\quad s_{2y}=\sum_{i=1}^{n}(y_i-\bar{y})(x_{i2}-\bar{x}_2)$

$r_{12}=\frac{s_{12}}{\sqrt{s_1^2s_2^2}}=s_{12}$. $\quad s_{12}=\sum_{i=1}^{n}(x_{i1}-\bar{x}_1)(x_{i2}-\bar{x}_2)$

(관찰1)

만약에 $x_1$과 $x_2$사이에 강한 선형관계가 있다면 $r_{12}\approx 1$ or $r_{12}\approx -1$

$\hat{\beta}_1$의 분모 = $1-r_{12}^2 \approx 0$

$\hat{\beta}_1$의 분자 $\approx 0$

$\to$ $\hat{\beta}_1$와 $\hat{\beta}_2$의 값이 불안정할것 같다. 

(관찰2)

$\hat{\beta}_1$, $\hat{\beta}_2$의 분산과 공분산을 구해보자. 

$\mbox{V}(\hat{\beta_1})=\frac{1}{1-r_{12}^2}\sigma^2$

$\mbox{V}(\hat{\beta_2})=\frac{1}{1-r_{12}^2}\sigma^2$

$\mbox{cov}\big(\hat{\beta}_1,\hat{\beta}_2\big)=\frac{-r_{12}}{1-r_{12}^2}\sigma^2$

(관찰3)

$\mbox{cor}\big(\hat{\beta}_1,\hat{\beta}_2\big)=-r_{12}$ 

In [2]:
import rpy2 
%load_ext rpy2.ipython

In [3]:
%%R 
set.seed(999)
n<-20
toeic<-750+rnorm(n,sd=80)
toeic[toeic>990]<-990
toeic<-round(toeic)
teps<-toeic - 10 + rnorm(n,sd=1)
gpa<-3.5+rnorm(n,sd=0.3)
gpa[gpa>4.5]<-4.5 
gpa<-round(gpa,1)
sal<-gpa*600+toeic*5+rnorm(n,sd=300)
sal<-round(sal)

In [4]:
%%R 
cor(toeic,teps)

[1] 0.9999459


(직관) 

토익이나 텝스점수나 그게 그거이다. (강한 상관관계)

원래모형은 `연봉=학점*600+토익*5+오차`인데, 토익이나 텝스나 그게 그거이므로, 아래와 같은 모형들도 거의 참모형이라고 생각할 수 있다. 

(1) `연봉=학점*600+토익*2+텝스*3+오차`

(2) `연봉=학점*600+토익*1+텝스*4+오차`

(3) `연봉=학점*600+토익*(-5)+텝스*10+오차`

(4) `연봉=학점*600+토익*(-1000)+텝스*(1005)+오차`

(5) `연봉=학점*600+토익*(-10000)+텝스*(10005)+오차`

...

결국에는 토익의 계수와 텝스의 계수를 더해서 5만되면 참모형

In [5]:
%%R 
lm1<-lm(sal~gpa+toeic+teps)
lm1


Call:
lm(formula = sal ~ gpa + toeic + teps)

Coefficients:
(Intercept)          gpa        toeic         teps  
   -1975.85       901.26        30.03       -24.08  



토익의 계수는 30.03, 텝스의 계수는 -24.08 로 추정되었다. 두개더하면 대략 5.  

몇번 더 시도를 해보자. 

(시도2)

In [5]:
%%R 
set.seed(1)
n<-20
toeic<-750+rnorm(n,sd=80)
toeic[toeic>990]<-990
toeic<-round(toeic)
teps<-toeic - 10 + rnorm(n,sd=1)
gpa<-3.5+rnorm(n,sd=0.3)
gpa[gpa>4.5]<-4.5 
gpa<-round(gpa,1)
sal<-gpa*600+toeic*5+rnorm(n,sd=300)
sal<-round(sal)

In [6]:
%%R 
lm1<-lm(sal~gpa+toeic+teps)
lm1


Call:
lm(formula = sal ~ gpa + toeic + teps)

Coefficients:
(Intercept)          gpa        toeic         teps  
    -194.35      1088.95       -80.35        84.48  



이번에는 토익의 계수는 -80.35, 텝스의 계수는 84.48. 두개 더하면 대충 5

(시도3)

In [7]:
%%R 
set.seed(2)
n<-20
toeic<-750+rnorm(n,sd=80)
toeic[toeic>990]<-990
toeic<-round(toeic)
teps<-toeic - 10 + rnorm(n,sd=1)
gpa<-3.5+rnorm(n,sd=0.3)
gpa[gpa>4.5]<-4.5 
gpa<-round(gpa,1)
sal<-gpa*600+toeic*5+rnorm(n,sd=300)
sal<-round(sal)

In [8]:
%%R 
lm1<-lm(sal~gpa+toeic+teps)
lm1


Call:
lm(formula = sal ~ gpa + toeic + teps)

Coefficients:
(Intercept)          gpa        toeic         teps  
    -1519.2        621.8        107.5       -102.0  



토익의 계수는 107.5, 텝스의 계수는 -102.0. 두개더하면 대충 5

(시도4)

In [9]:
%%R 
set.seed(3)
n<-20
toeic<-750+rnorm(n,sd=80)
toeic[toeic>990]<-990
toeic<-round(toeic)
teps<-toeic - 10 + rnorm(n,sd=1)
gpa<-3.5+rnorm(n,sd=0.3)
gpa[gpa>4.5]<-4.5 
gpa<-round(gpa,1)
sal<-gpa*600+toeic*5+rnorm(n,sd=300)
sal<-round(sal)

In [10]:
%%R 
lm1<-lm(sal~gpa+toeic+teps)
lm1


Call:
lm(formula = sal ~ gpa + toeic + teps)

Coefficients:
(Intercept)          gpa        toeic         teps  
     510.91       659.40       -35.18        39.86  



토익의 계수는 -35.18, 텝스의 계수는 39.86. 두개더하면 대충 5. 

*[다중공선성의 특징]*

(1) 추정하는 $\hat{\beta_1}$, $\hat{\beta_2}$가 어떤값일지 거의 예측안된다. 
- 5근처의 값이 나올때도 있고, 30근처의 값이 나오기도 하고, 100근처의 값이 나오기도 한다. 
- $\hat{\beta}_1$, $\hat{\beta}_2$의 분산이 크다. 

(2) 그래도 $\hat{\beta}_1+\hat{\beta}_2\approx 5$라는 공통점은 있음.  

### 수식적으로 그럴듯해 보여도, 모두 바람직한 모형은 아니다. 

아래는 모두 참모형이라고 생각되어지는 상황이다. 

(1) $\hat{\beta}_1=2$, $\hat{\beta}_2=3$

(2) $\hat{\beta}_1=5$, $\hat{\beta}_2=0$

(3) $\hat{\beta}_1=10$, $\hat{\beta}_2=-5$

모두 참모형에 가깝지만 상식적으로 (3)은 용납할 수 잆음. 

이유1: (3)번과 같은 형태를 허용하면 $\hat{\beta}_1=10000, ~ \hat{\beta}_2=-9995$ 와 같은식으로도 만들수 있음. 

이유2: (해석불가능한 모델)

$\hat{\beta}_2$가 의미하는 것은 텝스점수가 얼마나 연봉에 영향을 주는지이다. 

즉 텝스점수 1점을 올리면 연봉이 5만원 깍임. 

상식적으로 말이 안된다. 

### 해결책 

(3)과 같은 경우에 패널티를 주자!

In [11]:
%%R 
set.seed(2)
n<-20
toeic<-750+rnorm(n,sd=80)
toeic[toeic>990]<-990
toeic<-round(toeic)
teps<-toeic - 10 + rnorm(n,sd=1)
gpa<-3.5+rnorm(n,sd=0.3)
gpa[gpa>4.5]<-4.5 
gpa<-round(gpa,1)
sal<-gpa*600+toeic*5+rnorm(n,sd=300)
sal<-round(sal)

In [12]:
%%R 
lm(sal~gpa+toeic+teps)


Call:
lm(formula = sal ~ gpa + toeic + teps)

Coefficients:
(Intercept)          gpa        toeic         teps  
    -1519.2        621.8        107.5       -102.0  



$\hat{\beta}_0=-1519.2$ 

$\hat{\beta}_1=621.8$

$\hat{\beta}_2=107.5$

$\hat{\beta}_3=-102.0$

이 모형의 결과는 $L(\boldsymbol{\beta})$ 를 최소화한 $\beta$를 구한 결과임. 

In [13]:
%%R 
y<-sal
X<-cbind(1,gpa,toeic,teps)
L<-function(beta){
    t(y-X%*%beta)%*%(y-X%*%beta)
}

In [14]:
%%R 
#beta<-c(-1519.2,621.8,107.5,-102.0)
beta<-c(-1519.2,621.8,107.5,-102.0)
L(beta)/1529766

          [,1]
[1,] 0.9999998


아이디어: $L(\boldsymbol{\beta})$를 조금 바꾸자.

(1) $\hat{\beta}_1=2$, $\hat{\beta}_2=3$ $\longrightarrow$ $L(\boldsymbol{\beta})+ 10$ 

(2) $\hat{\beta}_1=5$, $\hat{\beta}_2=0$  $\longrightarrow$ $L(\boldsymbol{\beta})+ 20$ 

(3) $\hat{\beta}_1=10$, $\hat{\beta}_2=-5$ $\longrightarrow$ $L(\boldsymbol{\beta})+ 1000$ 

(4) $\hat{\beta}_1=100$, $\hat{\beta}_2=-105$ $\longrightarrow$ $L(\boldsymbol{\beta})+ 10000000$ 